In [ ]:
import os import sys import torch import logging import random import numpy as np import pandas as pd from datasets import load_dataset, Dataset, concatenate_datasets from transformers import ( AutoTokenizer, AutoModel, TrainingArguments, Trainer, EarlyStoppingCallback, ) from torch import nn from torch.utils.data import Dataset as TorchDataset from sklearn.metrics import ( classification_report, f1_score, accuracy_score, roc_auc_score, ) from sklearn.model_selection import train_test_split

In [ ]:
CONFIG = { "base_model":       "distilbert-base-uncased", "output_dir":       "./distilbert-classifier-checkpoints", "final_model_dir":  "./distilbert-emotion-risk-final", "backup_dir":       "./distilbert-classifier-backup", "log_file":         "training_classifier.log",
"max_length":       128,        # short — classifier doesn't need long context
"learning_rate":    2e-5,
"num_epochs":       4,
"batch_size":       32,         # larger batch = faster for small model
"eval_steps":       50,
"save_steps":       50,
"patience":         3,
"seed":             42,

# Class weights for imbalanced risk labels
"risk_pos_weight":  5.0,        # risk=1 is rare — upweight it

# Emotion labels — must match TinyLlama training tags
"emotion_labels": [
    "neutral", "joy", "sadness",
    "anger", "fear", "disgust", "surprise"
],
}

In [ ]:
##Risk keywords — used to auto-label risk=1 from text

In [ ]:
RISK_KEYWORDS = [ # Suicidal ideation "want to die", "kill myself", "end my life", "suicide", "don't want to live", "no reason to live", "better off dead", "can't go on", "not worth living", "end it all", # Self harm "hurt myself", "cut myself", "self harm", "self-harm", "harm myself", "punish myself", # Crisis signals "give up on everything", "nobody cares", "completely alone", "no hope", "hopeless", "nothing matters anymore", "can't take it anymore", "losing my mind", # Severe distress "falling apart", "breaking down", "can't cope", "don't see the point", "worthless", "useless", ]
logging.basicConfig( level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", handlers=[ logging.FileHandler(CONFIG["log_file"]), logging.StreamHandler(sys.stdout), ], ) log = logging.getLogger(name)

In [ ]:
##Check Gpu status

In [ ]:
def check_gpu(): 
    if not torch.cuda.is_available():
        log.warning("No GPU found — training will be slow on CPU")
        return "cpu" log.info(f"GPU  : {torch.cuda.get_device_name(0)}") log.info(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB") return "cuda"

In [ ]:
##Risk Labeller

In [ ]:
def label_risk(text: str) -> int: 
        """ Binary risk labeler. 0 = safe conversation 1 = at-risk (distress signals detected) """ 
        text_lower = text.lower()
        for kw in RISK_KEYWORDS:
            if kw in text_lower:
                return 1 
        return 0

In [ ]:
def map_emotion_to_id(emotion: str) -> int: 
        """Map emotion string to ID, with fuzzy matching."""
        emotion = emotion.lower().strip() # Direct match 
        if emotion in EMOTION2ID: 
            return EMOTION2ID[emotion] # Fuzzy mapping for dataset-specific labels 
        mapping = { "impressed":      "joy", "excited":        "joy", "grateful":       "joy", "confident":      "joy", "anticipating":   "joy", "joyful":         "joy", "caring":         "neutral", "faithful":       "neutral", "trusting":       "neutral", "devastated":     "sadness", "lonely":         "sadness", "guilty":         "sadness", "ashamed":        "sadness", "embarrassed":    "sadness", "jealous":        "anger", "furious":        "anger", "terrified":      "fear", "apprehensive":   "fear", "annoyed":        "anger", "disgusted":      "disgust", "surprised":      "surprise", }
        mapped = mapping.get(emotion, "neutral") 
        return EMOTION2ID[mapped]

In [ ]:
##Load Datasets

In [1]:
def load_empathetic_dialogues(): 
    """Load EmpatheticDialogues — has clean emotion labels.""" 
    log.info("Loading EmpatheticDialogues...") 
    try:
        ds = load_dataset( "facebook/empathetic_dialogues", trust_remote_code=True )
        records = []
        for split in ["train", "validation"]:
            for row in ds[split]:
                text    = str(row.get("prompt", "")).strip()
                emotion = str(row.get("context", "neutral")).strip()
                if not text or len(text) < 5:
                    continue
                records.append({ "text":     text, "emotion":  map_emotion_to_id(emotion), "risk":     label_risk(text), })
                log.info(f"EmpatheticDialogues: {len(records)} records")
                return records 
    except Exception as e:
        log.warning(f"EmpatheticDialogues failed: {e}")
        return []

SyntaxError: invalid non-printable character U+00A0 (884385826.py, line 1)

In [ ]:
def load_mental_health_data():
    """Load mental health datasets — good source of risk=1 examples."""
    log.info("Loading mental health datasets...")
    records = []

In [ ]:
datasets_to_try = [
    ("Amod/mental_health_counseling_conversations", "Context"),
    ("ShenLab/MentalChat16K", "input"),
    ("heliosbrahma/mental_health_chatbot_dataset", "input"),
]

for repo, col in datasets_to_try:
    try:
        ds = load_dataset(repo, split="train", trust_remote_code=True)
        for row in ds:
            text = str(row.get(col, "")).strip()
            if not text or len(text) < 5:
                continue
            risk = label_risk(text)
            # Infer emotion from keywords
            emotion = infer_emotion_from_text(text)
            records.append({
                "text":    text,
                "emotion": emotion,
                "risk":    risk,
            })
        log.info(f"{repo}: {len(ds)} records ✅")
    except Exception as e:
        log.warning(f"{repo} failed: {e}")

log.info(f"Mental health total: {len(records)} records")
return records

In [ ]:
def infer_emotion_from_text(text: str) -> int:
    """Simple keyword-based emotion inference for unlabeled text.""" 
    text = text.lower()
    if any(w in text for w in ["happy", "excited", "wonderful", "joy", "great", "amazing", "proud", "thrilled"]):
        return EMOTION2ID["joy"] 
    if any(w in text for w in ["sad", "depress", "cry", "lonely", "hopeless", "grief", "empty", "loss"]):
        return EMOTION2ID["sadness"]
    if any(w in text for w in ["angry", "anger", "furious", "frustrated", "rage", "irritated", "annoyed"]):
        return EMOTION2ID["anger"]
    if any(w in text for w in ["scared", "afraid", "anxious", "anxiety", "fear", "panic", "worry", "nervous", "overwhelm", "terrified"]):
        return EMOTION2ID["fear"]
    if any(w in text for w in ["disgust", "sick", "gross", "repuls"]): 
        return EMOTION2ID["disgust"] 
    if any(w in text for w in ["surprise", "shock", "unexpected", "sudden"]): 
        return EMOTION2ID["surprise"] 
    return EMOTION2ID["neutral"]

In [ ]:
##Load and Combine data

In [ ]:
def load_and_combine_data():
    """Load all datasets, combine, balance, and split.""" 
    empathetic = load_empathetic_dialogues() 
    mental     = load_mental_health_data()
    all_records = empathetic + mental

    if len(all_records) == 0:
        raise RuntimeError("No data loaded. Check internet connection.")

    df = pd.DataFrame(all_records)
    df = df.drop_duplicates(subset=["text"])
    df = df[df["text"].str.len() > 10]

    # Log class distribution
    log.info(f"\nTotal records: {len(df)}")
    log.info(f"Emotion distribution:\n{df['emotion'].value_counts()}")
    log.info(f"Risk distribution:\n{df['risk'].value_counts()}")
    log.info(f"Risk positive rate: {df['risk'].mean():.2%}")

    # Balance risk classes — oversample risk=1
    risk_positive = df[df["risk"] == 1]
    risk_negative = df[df["risk"] == 0]

    if len(risk_positive) < len(risk_negative) * 0.1:
        # Oversample risk=1 to at least 10% of dataset
        target = int(len(risk_negative) * 0.15)
        risk_positive_oversampled = risk_positive.sample(
            n=target, replace=True, random_state=CONFIG["seed"]
        )
        df = pd.concat([risk_negative, risk_positive_oversampled])
        df = df.sample(frac=1, random_state=CONFIG["seed"]).reset_index(drop=True)
        log.info(f"After oversampling: {len(df)} records")
        log.info(f"New risk rate: {df['risk'].mean():.2%}")

    # Train / val / test split — 80/10/10
    train_df, temp_df = train_test_split(
        df, test_size=0.2, random_state=CONFIG["seed"],
        stratify=df["risk"]
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.5, random_state=CONFIG["seed"],
        stratify=temp_df["risk"]
    )

    log.info(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    return train_df, val_df, test_df

In [ ]:
##Loading Pytorch

In [ ]:
class EmotionRiskDataset(TorchDataset): 
    def init(self, df: pd.DataFrame, tokenizer, max_length: int):
        self.texts = df["text"].tolist() 
        self.emotions = df["emotion"].tolist() 
        self.risks = df["risk"].tolist() 
        self.tokenizer  = tokenizer 
        self.max_length = max_length
    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
    return {
        "input_ids":      encoding["input_ids"].squeeze(),
        "attention_mask": encoding["attention_mask"].squeeze(),
        "emotion_labels": torch.tensor(self.emotions[idx], dtype=torch.long),
        "risk_labels":    torch.tensor(self.risks[idx],    dtype=torch.float),
    }

In [ ]:
##Multiclass Classifier

In [ ]:
class EmotionRiskClassifier(nn.Module):
    """
    DistilBERT with two heads:
      Head 1 — Emotion classifier (7 classes, CrossEntropy)
      Head 2 — Risk detector (binary, BCEWithLogits)
 
    Shared DistilBERT backbone — efficient for edge deployment.
    """
 
    def __init__(
        self,
        model_name: str,
        num_emotions: int,
        risk_pos_weight: float,
        dropout: float = 0.3,
    ):
        super().__init__()
 
        self.distilbert = AutoModel.from_pretrained(model_name)
        hidden_size = self.distilbert.config.hidden_size  # 768
 
        # Shared intermediate layer
        self.shared_layer = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(512),
        )
 
        # Head 1 — Emotion (7-class)
        self.emotion_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_emotions),
        )
 
        # Head 2 — Risk (binary)
        self.risk_head = nn.Sequential(
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )
 
        # Loss functions
        self.emotion_loss_fn = nn.CrossEntropyLoss()
        self.risk_loss_fn    = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([risk_pos_weight])
        )
 
        # Loss weighting — risk detection slightly upweighted
        self.emotion_weight = 0.6
        self.risk_weight    = 0.4
 
    def forward(
        self,
        input_ids,
        attention_mask,
        emotion_labels=None,
        risk_labels=None,
    ):
        # DistilBERT encoding
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
 
        # CLS token representation
        cls_output = outputs.last_hidden_state[:, 0, :]  # [batch, 768]
 
        # Shared layer
        shared = self.shared_layer(cls_output)           # [batch, 512]
 
        # Head outputs
        emotion_logits = self.emotion_head(shared)       # [batch, 7]
        risk_logits    = self.risk_head(shared).squeeze(-1)  # [batch]
 
        output = {
            "emotion_logits": emotion_logits,
            "risk_logits":    risk_logits,
        }
 
        # Compute loss if labels provided
        if emotion_labels is not None and risk_labels is not None:
            emotion_loss = self.emotion_loss_fn(emotion_logits, emotion_labels)
            risk_loss    = self.risk_loss_fn(
                risk_logits,
                risk_labels.to(risk_logits.device)
            )
            # Combined weighted loss
            total_loss = (
                self.emotion_weight * emotion_loss +
                self.risk_weight    * risk_loss
            )
            output["loss"]         = total_loss
            output["emotion_loss"] = emotion_loss
            output["risk_loss"]    = risk_loss
 
        return output


In [ ]:
##Custom Trainer

In [ ]:
class MultiTaskTrainer(Trainer):
    """
    Custom Trainer that handles the multitask model's
    output format and combined loss.
    """
 
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            emotion_labels=inputs["emotion_labels"],
            risk_labels=inputs["risk_labels"],
        )
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss

In [ ]:
##Metrics

In [ ]:
def compute_metrics(eval_pred):
    """
    Compute emotion accuracy + risk F1/AUC.
    eval_pred contains raw model outputs — need custom parsing.
    """
    # Note: with custom model output dict, predictions come as tuple
    predictions, labels = eval_pred
 
    if isinstance(predictions, tuple):
        emotion_logits = predictions[0]
        risk_logits    = predictions[1]
    else:
        return {}
 
    # Emotion predictions
    emotion_preds = np.argmax(emotion_logits, axis=1)
    emotion_true  = labels[:, 0].astype(int)
 
    # Risk predictions
    risk_probs  = 1 / (1 + np.exp(-risk_logits))   # sigmoid
    risk_preds  = (risk_probs > 0.5).astype(int)
    risk_true   = labels[:, 1].astype(int)
 
    emotion_acc = accuracy_score(emotion_true, emotion_preds)
    emotion_f1  = f1_score(emotion_true, emotion_preds,
                            average="weighted", zero_division=0)
 
    risk_f1  = f1_score(risk_true, risk_preds, zero_division=0)
    try:
        risk_auc = roc_auc_score(risk_true, risk_probs)
    except Exception:
        risk_auc = 0.5
 
    return {
        "emotion_accuracy": round(emotion_acc, 4),
        "emotion_f1":       round(emotion_f1,  4),
        "risk_f1":          round(risk_f1,     4),
        "risk_auc":         round(risk_auc,    4),
    }


In [ ]:
##Train

In [ ]:
def train(model, tokenizer, train_df, val_df):
    train_dataset = EmotionRiskDataset(train_df, tokenizer, CONFIG["max_length"])
    val_dataset   = EmotionRiskDataset(val_df,   tokenizer, CONFIG["max_length"])
 
    training_args = TrainingArguments(
        output_dir=CONFIG["output_dir"],
        num_train_epochs=CONFIG["num_epochs"],
        per_device_train_batch_size=CONFIG["batch_size"],
        per_device_eval_batch_size=CONFIG["batch_size"],
        learning_rate=CONFIG["learning_rate"],
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="cosine",
        evaluation_strategy="steps",
        eval_steps=CONFIG["eval_steps"],
        save_strategy="steps",
        save_steps=CONFIG["save_steps"],
        save_total_limit=3,
        logging_steps=20,
        load_best_model_at_end=True,
        metric_for_best_model="risk_f1",  # optimize for risk detection
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=CONFIG["seed"],
        remove_unused_columns=False,      # important for custom dataset
    )
 
    trainer = MultiTaskTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=CONFIG["patience"],
                early_stopping_threshold=0.005,
            )
        ],
    )
 
    log.info(f"Train samples : {len(train_dataset)}")
    log.info(f"Val samples   : {len(val_dataset)}")
    log.info("Starting classifier training...")
 
    trainer.train()
    log.info("Classifier training complete ✅")
    return trainer

In [ ]:
##Evaluation

In [ ]:
def evaluate_model(model, tokenizer, test_df, device):
    log.info("\n" + "="*60)
    log.info("FINAL EVALUATION ON TEST SET")
    log.info("="*60)
 
    model.eval()
    dataset = EmotionRiskDataset(test_df, tokenizer, CONFIG["max_length"])
 
    emotion_preds_list = []
    emotion_true_list  = []
    risk_preds_list    = []
    risk_probs_list    = []
    risk_true_list     = []
 
    from torch.utils.data import DataLoader
    loader = DataLoader(dataset, batch_size=64, shuffle=False)
 
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
 
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
 
            emotion_logits = outputs["emotion_logits"].cpu().numpy()
            risk_logits    = outputs["risk_logits"].cpu().numpy()
 
            emotion_preds = np.argmax(emotion_logits, axis=1)
            risk_probs    = 1 / (1 + np.exp(-risk_logits))
            risk_preds    = (risk_probs > 0.5).astype(int)
 
            emotion_preds_list.extend(emotion_preds)
            emotion_true_list.extend(batch["emotion_labels"].numpy())
            risk_preds_list.extend(risk_preds)
            risk_probs_list.extend(risk_probs)
            risk_true_list.extend(batch["risk_labels"].numpy())
 
    # Emotion report
    log.info("\n--- EMOTION CLASSIFICATION ---")
    emotion_names = CONFIG["emotion_labels"]
    log.info("\n" + classification_report(
        emotion_true_list, emotion_preds_list,
        target_names=emotion_names,
        zero_division=0
    ))
 
    # Risk report
    log.info("\n--- RISK DETECTION ---")
    log.info("\n" + classification_report(
        risk_true_list, risk_preds_list,
        target_names=["safe", "at-risk"],
        zero_division=0
    ))
 
    try:
        auc = roc_auc_score(risk_true_list, risk_probs_list)
        log.info(f"Risk AUC-ROC: {auc:.4f}")
    except Exception:
        pass
 
    overall_emotion_acc = accuracy_score(emotion_true_list, emotion_preds_list)
    risk_f1 = f1_score(risk_true_list, risk_preds_list, zero_division=0)
 
    log.info(f"\nEmotion Accuracy : {overall_emotion_acc:.4f}")
    log.info(f"Risk F1 Score    : {risk_f1:.4f}")
 
    return overall_emotion_acc, risk_f1


In [ ]:
##Save Model

In [ ]:
def save_model(model, tokenizer):
    os.makedirs(CONFIG["final_model_dir"], exist_ok=True)
 
    # Save DistilBERT backbone + tokenizer
    model.distilbert.save_pretrained(CONFIG["final_model_dir"])
    tokenizer.save_pretrained(CONFIG["final_model_dir"])
 
    # Save full model state dict (includes both heads)
    torch.save(
        model.state_dict(),
        os.path.join(CONFIG["final_model_dir"], "classifier_weights.pt")
    )
 
    # Save config for inference
    import json
    config = {
        "emotion_labels":    CONFIG["emotion_labels"],
        "max_length":        CONFIG["max_length"],
        "base_model":        CONFIG["base_model"],
        "num_emotions":      len(CONFIG["emotion_labels"]),
        "risk_pos_weight":   CONFIG["risk_pos_weight"],
        "risk_threshold":    0.5,
    }
    with open(os.path.join(CONFIG["final_model_dir"], "classifier_config.json"), "w") as f:
        json.dump(config, f, indent=2)
 
    log.info(f"Model saved to {CONFIG['final_model_dir']} ✅")


In [ ]:
##Inference Pipeline

In [ ]:
class EmotionRiskPipeline:
    """
    Ready-to-use inference pipeline.
    Loads saved model and runs predictions.
    Use this in your Flutter backend / edge server.
    """
 
    def __init__(self, model_dir: str, device: str = "cpu"):
        import json
 
        self.device = torch.device(device)
 
        # Load config
        with open(os.path.join(model_dir, "classifier_config.json")) as f:
            self.config = json.load(f)
 
        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
 
        # Rebuild model
        self.model = EmotionRiskClassifier(
            model_name=model_dir,
            num_emotions=self.config["num_emotions"],
            risk_pos_weight=self.config["risk_pos_weight"],
        )
 
        # Load weights
        weights_path = os.path.join(model_dir, "classifier_weights.pt")
        self.model.load_state_dict(
            torch.load(weights_path, map_location=self.device)
        )
        self.model.to(self.device)
        self.model.eval()
 
        self.emotion_labels = self.config["emotion_labels"]
        self.risk_threshold = self.config["risk_threshold"]
 
        log.info(f"Pipeline loaded from {model_dir} ✅")
 
    def predict(self, text: str) -> dict:
        """
        Returns:
        {
            "emotion":          "sadness",
            "emotion_id":       2,
            "emotion_scores":   {"neutral": 0.05, "joy": 0.02, ...},
            "risk":             1,
            "risk_probability": 0.87,
            "risk_label":       "at-risk"
        }
        """
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.config["max_length"],
            padding="max_length",
            return_tensors="pt",
        ).to(self.device)
 
        with torch.no_grad():
            outputs = self.model(
                input_ids=encoding["input_ids"],
                attention_mask=encoding["attention_mask"],
            )
 
        # Emotion
        emotion_logits = outputs["emotion_logits"][0]
        emotion_probs  = torch.softmax(emotion_logits, dim=0).cpu().numpy()
        emotion_id     = int(np.argmax(emotion_probs))
        emotion_label  = self.emotion_labels[emotion_id]
        emotion_scores = {
            label: round(float(prob), 4)
            for label, prob in zip(self.emotion_labels, emotion_probs)
        }
 
        # Risk
        risk_logit  = outputs["risk_logits"][0].item()
        risk_prob   = float(1 / (1 + np.exp(-risk_logit)))
        risk_binary = 1 if risk_prob >= self.risk_threshold else 0
        risk_label  = "at-risk" if risk_binary == 1 else "safe"
 
        return {
            "emotion":          emotion_label,
            "emotion_id":       emotion_id,
            "emotion_scores":   emotion_scores,
            "risk":             risk_binary,
            "risk_probability": round(risk_prob, 4),
            "risk_label":       risk_label,
        }
 
    def predict_batch(self, texts: list) -> list:
        """Run predictions on a list of texts."""
        return [self.predict(t) for t in texts]


In [ ]:
##Inference test

In [ ]:
def run_inference_tests(pipeline: EmotionRiskPipeline):
    log.info("\n" + "="*60)
    log.info("INFERENCE TESTS")
    log.info("="*60)
 
    test_cases = [
        # Expected: emotion, risk=0
        ("I got promoted today, I can't believe it!",            "joy",     0),
        ("I've been feeling really lonely lately.",              "sadness", 0),
        ("My boss blamed me for something I didn't do!",         "anger",   0),
        ("I have a big exam tomorrow and I'm terrified.",        "fear",    0),
        ("I just bumped into my ex at the mall.",                "surprise",0),
        ("What did you do over the weekend?",                    "neutral", 0),
        # Expected: risk=1
        ("I want to end my life, I can't take it anymore.",      "sadness", 1),
        ("I've been cutting myself to cope with the pain.",      "sadness", 1),
        ("There's no point in living, I give up on everything.", "sadness", 1),
        ("I feel completely worthless and hopeless.",            "sadness", 1),
        # Edge cases
        ("I'm so excited about my new job!",                     "joy",     0),
        ("I can't stop crying and I don't know why.",            "sadness", 0),
    ]
 
    correct_emotion = 0
    correct_risk    = 0
 
    for text, expected_emotion, expected_risk in test_cases:
        result = pipeline.predict(text)
 
        emotion_correct = result["emotion"] == expected_emotion
        risk_correct    = result["risk"] == expected_risk
 
        if emotion_correct: correct_emotion += 1
        if risk_correct:    correct_risk    += 1
 
        emotion_flag = "✅" if emotion_correct else "❌"
        risk_flag    = "✅" if risk_correct    else "⚠️"
 
        log.info(f"\n📝 {text[:60]}...")
        log.info(f"   Emotion : {emotion_flag} {result['emotion']:10s} "
                 f"(expected: {expected_emotion})")
        log.info(f"   Risk    : {risk_flag} {result['risk']} [{result['risk_label']:8s}] "
                 f"prob={result['risk_probability']:.3f} "
                 f"(expected: {expected_risk})")
        log.info(f"   Scores  : {result['emotion_scores']}")
 
    n = len(test_cases)
    log.info(f"\n{'='*60}")
    log.info(f"Emotion accuracy : {correct_emotion}/{n} = {correct_emotion/n:.1%}")
    log.info(f"Risk accuracy    : {correct_risk}/{n}    = {correct_risk/n:.1%}")


In [ ]:
##Export to ONNX

In [ ]:
def export_to_onnx(model, tokenizer, device):
    """
    Export to ONNX for edge deployment.
    Smaller and faster than PyTorch on CPU.
    """
    log.info("Exporting to ONNX...")
 
    try:
        dummy_input = tokenizer(
            "I feel so happy today",
            return_tensors="pt",
            max_length=CONFIG["max_length"],
            padding="max_length",
            truncation=True,
        ).to(device)
 
        onnx_path = os.path.join(CONFIG["final_model_dir"], "classifier.onnx")
 
        torch.onnx.export(
            model,
            (dummy_input["input_ids"], dummy_input["attention_mask"]),
            onnx_path,
            input_names=["input_ids", "attention_mask"],
            output_names=["emotion_logits", "risk_logits"],
            dynamic_axes={
                "input_ids":      {0: "batch_size"},
                "attention_mask": {0: "batch_size"},
                "emotion_logits": {0: "batch_size"},
                "risk_logits":    {0: "batch_size"},
            },
            opset_version=14,
            do_constant_folding=True,
        )
 
        size_mb = os.path.getsize(onnx_path) / 1e6
        log.info(f"ONNX exported: {onnx_path} ({size_mb:.1f} MB) ✅")
        log.info("Use ONNX Runtime for 2-3x faster edge inference")
 
    except Exception as e:
        log.warning(f"ONNX export failed: {e}")
        log.warning("PyTorch model still saved and usable")


In [ ]:
##Maindef 
main():
    log.info("=" * 60)
    log.info("Multilabel Emotion + Risk Classifier")
    log.info("Base: DistilBERT — fine-tuned for edge deployment") log.info("=" * 60)
    # 1. GPU
    device = check_gpu()

    # 2. Load data
    train_df, val_df, test_df = load_and_combine_data()

    # 3. Load tokenizer
    log.info(f"Loading tokenizer: {CONFIG['base_model']}")
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])

    # 4. Build model
    log.info("Building EmotionRiskClassifier...")
    model = EmotionRiskClassifier(
        model_name=CONFIG["base_model"],
        num_emotions=len(CONFIG["emotion_labels"]),
        risk_pos_weight=CONFIG["risk_pos_weight"],
    )
    model.to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info(f"Total params     : {total_params:,}")
    log.info(f"Trainable params : {trainable_params:,}")
        # 5. Train
    trainer = train(model, tokenizer, train_df, val_df)
 
    # 6. Evaluate on test set
    evaluate_model(trainer.model, tokenizer, test_df, device)
 
    # 7. Save
    save_model(trainer.model, tokenizer)
 
    # 8. ONNX export
    export_to_onnx(trainer.model, tokenizer, device)
 
    # 9. Load pipeline and test
    log.info("\nLoading inference pipeline for final test...")
    pipeline = EmotionRiskPipeline(
        model_dir=CONFIG["final_model_dir"],
        device=device,
    )
    run_inference_tests(pipeline)
 
    log.info("\n" + "="*60)
    log.info("✅ Classifier training complete!")
    log.info(f"Model saved : {CONFIG['final_model_dir']}")
    log.info(f"ONNX model  : {CONFIG['final_model_dir']}/classifier.onnx")
    log.info(f"Logs        : {CONFIG['log_file']}")
    log.info("="*60)
 